# Indic Deepfake Speech Detection - OOF + LoRA Pipeline

Binary classifier: **genuine human speech (0)** vs **TTS-synthesized (1)** across 16 Indian languages.

## System Design

```
TRAINING
--------
1. Decode + resample every clip ONCE into an int16 waveform cache (reused everywhere).
2. Extract handcrafted acoustic features (36-d) from the cache - parallel across CPU cores.
3. 5-fold OOF Wav2Vec2 fine-tuning with LoRA (fp16 AMP):
     Fold k -> fine-tune on 4 folds -> extract embeddings + NN probabilities for:
       - the held-out fold k (leak-free OOF features, never seen by this model)
       - the test set (cheap: 2.6k clips; the 5 fold models form a free ensemble)
4. Train XGBoost on [OOF embeddings | handcrafted | metadata].

INFERENCE
---------
5. XGBoost predicts on each fold's test embeddings (averaged), blended with the
   NN ensemble probability. Blend weight is selected on the validation split.
```

> **Kaggle setup:** enable **GPU** (Settings -> Accelerator) and **Internet** (to download the dataset and Wav2Vec2).

In [ ]:
!pip uninstall -y torchao
!pip install -q "peft>=0.10" "datasets[audio]>=2.18" "transformers>=4.45" "librosa>=0.10.2" "xgboost>=2.1" soundfile

In [ ]:
import gc, json, warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import librosa
from tqdm.auto import tqdm
from datasets import load_dataset
from transformers import Wav2Vec2Model, Wav2Vec2Processor
from peft import LoraConfig, get_peft_model
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import (roc_auc_score, average_precision_score,
                              accuracy_score, f1_score)
from xgboost import XGBClassifier

warnings.filterwarnings("ignore")
torch.manual_seed(42)
np.random.seed(42)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

# ─── Hugging Face Hub Login ──────────────────────────────────────────
import os
from huggingface_hub import login

# Set your Hugging Face token here (or leave as 'your_hugging_face_token_here' to load from environment/secrets)
HF_TOKEN = 'your_hugging_face_token_here'

token_to_use = HF_TOKEN
if not token_to_use or token_to_use == "your_hugging_face_token_here":
    try:
        from kaggle_secrets import UserSecretsClient
        user_secrets = UserSecretsClient()
        token_to_use = user_secrets.get_secret("HF_TOKEN") or user_secrets.get_secret("HUGGING_FACE_HUB_TOKEN")
    except Exception:
        token_to_use = os.environ.get("HF_TOKEN", "") or os.environ.get("HUGGING_FACE_HUB_TOKEN", "")

if token_to_use and token_to_use != "your_hugging_face_token_here":
    try:
        login(token=token_to_use)
        print("Successfully authenticated with Hugging Face Hub.")
    except Exception as e:
        print(f"Failed to authenticate with Hugging Face Hub: {e}")
else:
    print("No Hugging Face token found. Gated models (like 'ai4bharat/indicwav2vec-hindi') will fall back to public mirrors if unauthorized.")


## Configuration
All knobs in one place. Use `TRAIN_LIMIT` / `TEST_LIMIT` for a fast smoke run.

In [ ]:
# ── Dataset ───────────────────────────────────────────────────────────────────
DATASET_NAME      = "/kaggle/input/datasets/iveeaten3223times/multilingual-indian-speech-data"
EXTRA_DATA_DIR    = "/kaggle/input/datasets/adhithyasash1/indic-deepfake-challenge-extra-dataset/extra-dataset"  # Path to merged extra dataset, or None to skip
SPEECH_MODEL_NAME = "ai4bharat/indicwav2vec-hindi"
# Alternatives worth trying (all dims are auto-derived, so swapping is a one-line change):
#   "facebook/wav2vec2-xls-r-300m"  - multilingual SSL (128 langs incl. Indic), the standard
#                                     anti-spoofing backbone in ASVspoof literature
#   "apoorva-ak/indicwav2vec_base"  - public non-gated Indic mirror (smaller, faster)
#   "facebook/wav2vec2-base-960h"   - small English baseline for smoke runs

# ── Audio ─────────────────────────────────────────────────────────────────────
TARGET_SR    = 16_000
MAX_SECONDS  = 5.0
MAX_SAMPLES  = int(TARGET_SR * MAX_SECONDS)
PITCH_METHOD = "yin"        # "yin" = fast; "pyin" = more accurate

# ── LoRA fine-tuning ──────────────────────────────────────────────────────────
N_FOLDS         = 5
LORA_RANK       = 16
LORA_ALPHA      = 32
LORA_DROPOUT    = 0.05      # Lowered from 0.1 for better speech embedding adaptation
LORA_LR         = 1e-4      # peak LR for the OneCycle schedule
N_EPOCHS        = 3
FINETUNE_BATCH  = 16        # DataParallel splits across GPUs (8/GPU on T4x2); halve if OOM
EXTRACT_BATCH   = 64        # inference only, no gradients - can be much larger
USE_AMP         = True      # fp16 autocast: ~2-3x faster on T4/P100 tensor cores
GRAD_CHECKPOINT = False     # not needed at 5 s clips / these batch sizes; enable only if OOM
EMBED_DIM       = 768       # placeholder - overwritten with the real backbone hidden_size
                            # at model-load time (indicwav2vec-hindi is a LARGE model: 1024)

# ── XGBoost ───────────────────────────────────────────────────────────────────
XGB_PARAMS = dict(
    learning_rate=0.02,     # Lowered for stable step sizes in 3K+ features
    n_estimators=1500,      # Increased to allow slow, precise boosting
    max_depth=6,            # Reduced from 7 to mitigate overfitting on high-dimensional vectors
    min_child_weight=3,     # Increased to prune splits on noisy/sparse embedding dimensions
    gamma=0.2,              # Increased to penalize weak tree nodes
    subsample=0.7,          # Subsampling rows prevents tree correlation
    colsample_bytree=0.6,   # Subsampling features prevents over-relying on specific embeddings
    reg_alpha=1.0,          # L1 regularization to encourage feature sparsity
    reg_lambda=2.0,         # L2 regularization to shrink correlated embedding weights
    tree_method="hist",
    objective="binary:logistic", random_state=42, eval_metric="auc",
)
XGB_EARLY_STOP = 50
XGB_VAL_SIZE   = 0.15
XGB_DEVICE     = "cuda" if DEVICE.type == "cuda" else "cpu"

# ── Quick-check (None = full run) ─────────────────────────────────────────────
TRAIN_LIMIT = None
TEST_LIMIT  = None
SEED        = 42

## Load Dataset

In [ ]:
import os
from datasets import load_dataset, Dataset, concatenate_datasets, Audio

def load_splits(train_limit=None, test_limit=None, extra_data_dir=None):
    is_local = DATASET_NAME.startswith("/")
    if is_local:
        # Load local main dataset from Kaggle using direct CSV loading
        metadata_dir = os.path.join(DATASET_NAME, "metadata")
        audio_dir    = os.path.join(DATASET_NAME, "audio")
        
        train_df = pd.read_csv(os.path.join(metadata_dir, "train.csv"))
        test_df  = pd.read_csv(os.path.join(metadata_dir, "test.csv"))
        
        # Map the nested CSV paths to the actual flat local file paths
        train_df["audio"] = train_df["audio_path"].apply(lambda p: os.path.join(audio_dir, os.path.basename(p)))
        test_df["audio"]  = test_df["audio_path"].apply(lambda p: os.path.join(audio_dir, os.path.basename(p)))
        
        if train_limit:
            train_df = train_df.iloc[:train_limit]
        if test_limit:
            test_df = test_df.iloc[:test_limit]
            
        tr = Dataset.from_pandas(train_df)
        te = Dataset.from_pandas(test_df)
        
        tr = tr.cast_column("audio", Audio(sampling_rate=16000))
        te = te.cast_column("audio", Audio(sampling_rate=16000))
    else:
        # Load from Hugging Face Hub
        if train_limit or test_limit:
            tr = load_dataset(DATASET_NAME, split=f"train[:{train_limit}]" if train_limit else "train")
            te = load_dataset(DATASET_NAME, split=f"test[:{test_limit}]"  if test_limit  else "test")
        else:
            ds = load_dataset(DATASET_NAME)
            tr, te = ds["train"], ds["test"]
        
    if extra_data_dir:
        print(f"Loading additional dataset from: {extra_data_dir}")
        extra_ds = load_dataset("audiofolder", data_dir=extra_data_dir, split="train")
        extra_ds = extra_ds.cast_column("audio", tr.features["audio"])
        extra_ds = extra_ds.select_columns(["text", "id", "language", "is_tts", "audio"])
        tr = concatenate_datasets([tr, extra_ds])
        
    return tr, te

train_ds, test_ds = load_splits(TRAIN_LIMIT, TEST_LIMIT, extra_data_dir=EXTRA_DATA_DIR)
print(f"Train: {len(train_ds)}   Test: {len(test_ds)}")

train_records = [dict(train_ds[i]) for i in range(len(train_ds))]
test_records  = [dict(test_ds[i])  for i in range(len(test_ds))]
train_labels  = np.array([r["is_tts"] for r in train_records], dtype=np.int64)
print("Positive (TTS) rate:", round(train_labels.mean(), 4))

## Audio Preprocessing

Every clip → mono → 16 kHz resample → peak-normalize → center-crop or zero-pad to 5 s.
Original duration is captured before cropping (used as a feature).

In [ ]:
def coerce_audio(audio):
    # Handle Hugging Face datasets v4 AudioDecoder objects dynamically
    if type(audio).__name__ == "AudioDecoder" or hasattr(audio, "get_all_samples"):
        samples = audio.get_all_samples()
        return np.asarray(samples.data), int(samples.sample_rate)
        
    if isinstance(audio, dict):
        if "array" in audio:
            return np.asarray(audio["array"]), int(audio["sampling_rate"])
        if "path" in audio:
            import soundfile as sf
            w, sr = sf.read(audio["path"], always_2d=False)
            return np.asarray(w), int(sr)
    if hasattr(audio, "array"):
        return np.asarray(audio.array), int(audio.sampling_rate)
    raise TypeError(f"Unsupported audio type: {type(audio)!r}")


def preprocess_waveform(audio):
    waveform, sr = coerce_audio(audio)
    waveform = np.asarray(waveform, dtype=np.float32)
    if waveform.ndim == 2:
        waveform = waveform.mean(axis=0 if waveform.shape[0] <= waveform.shape[1] else 1)
    waveform = waveform.reshape(-1)
    if waveform.size == 0:
        return np.zeros(MAX_SAMPLES, dtype=np.float32), 0.0
    if sr != TARGET_SR:
        waveform = librosa.resample(waveform, orig_sr=sr, target_sr=TARGET_SR).astype(np.float32)
    original_duration = float(waveform.size) / TARGET_SR
    peak = float(np.max(np.abs(waveform)))
    if peak > 0:
        waveform /= peak
    if waveform.size > MAX_SAMPLES:
        start = (waveform.size - MAX_SAMPLES) // 2
        waveform = waveform[start:start + MAX_SAMPLES]
    else:
        waveform = np.pad(waveform, (0, MAX_SAMPLES - waveform.size))
    return waveform.astype(np.float32), original_duration

## Waveform Cache - Decode Once, Reuse Everywhere

Audio decode + resample dominated CPU time: previously every clip was re-decoded on
every epoch of every fold (~19x per clip across the whole pipeline). Cache all clips
once as int16 PCM (5 s x 16 kHz x 2 B = 160 KB/clip, ~5.3 GB for train) in a
disk-backed memmap. int16 is standard PCM fidelity, so no meaningful signal loss.

In [ ]:
import os

def build_wave_cache(records, prefix):
    """Decode + resample + normalize + crop every clip exactly once, store as int16 PCM."""
    wpath, dpath = f"{prefix}_waves_i16.npy", f"{prefix}_durs.npy"
    if os.path.exists(wpath) and os.path.exists(dpath):
        waves = np.load(wpath, mmap_mode="r")
        if waves.shape == (len(records), MAX_SAMPLES):
            print(f"Using cached waveforms: {wpath} {waves.shape}")
            return waves, np.load(dpath)
        print(f"Stale waveform cache {waves.shape} - rebuilding.")

    waves = np.lib.format.open_memmap(
        wpath, mode="w+", dtype=np.int16, shape=(len(records), MAX_SAMPLES))
    durs = np.zeros(len(records), dtype=np.float32)
    for i, r in enumerate(tqdm(records, desc=f"caching {prefix} waveforms")):
        w, d = preprocess_waveform(r["audio"])     # float32 in [-1, 1] (peak-normalized)
        waves[i] = np.clip(w * 32767.0, -32768, 32767).astype(np.int16)
        durs[i]  = d
    waves.flush()
    np.save(dpath, durs)
    return np.load(wpath, mmap_mode="r"), durs

train_waves, train_durs = build_wave_cache(train_records, "train")
test_waves,  test_durs  = build_wave_cache(test_records,  "test")

## Handcrafted Acoustic Features (36-d)

Purpose-built anti-spoofing cues targeting what TTS systems get wrong:
pitch regularity, unnatural phase continuity, harmonic balance, and over-stable formants.

In [ ]:
HANDCRAFTED_NAMES = (
    ["f0_mean", "f0_std", "spectral_flux", "phase_coherence",
     "hnr", "rms_mean", "rms_std", "flatness_mean", "flatness_std", "duration_s"]
    + [f"mfcc_{i:02d}_mean" for i in range(13)]
    + [f"mfcc_{i:02d}_std"  for i in range(13)]
)

def extract_handcrafted(waveform, duration, sr=TARGET_SR):
    hop  = 512
    n_fft = min(2048, max(256, 2 ** int(np.floor(np.log2(max(256, waveform.size))))))

    try:
        if PITCH_METHOD == "yin":
            f0 = librosa.yin(waveform, fmin=librosa.note_to_hz("C2"),
                             fmax=librosa.note_to_hz("C7"), sr=sr, frame_length=n_fft)
        else:
            f0, _, _ = librosa.pyin(waveform, fmin=librosa.note_to_hz("C2"),
                                    fmax=librosa.note_to_hz("C7"), sr=sr, frame_length=n_fft)
        f0 = f0[~np.isnan(f0)]
    except Exception:
        f0 = np.array([0.0])
    if f0.size == 0:
        f0 = np.array([0.0])

    stft  = librosa.stft(waveform, n_fft=n_fft, hop_length=hop)
    mag   = np.abs(stft)

    if mag.shape[1] > 1:
        norm  = mag / (mag.sum(axis=0, keepdims=True) + 1e-10)
        flux  = float(np.mean(np.sqrt(np.sum(np.diff(norm, axis=1) ** 2, axis=0))))
    else:
        flux  = 0.0

    pd_   = np.diff(np.angle(stft), axis=1)
    phase_coh = 0.0 if pd_.size == 0 else float(
        np.mean(np.abs(np.mean(np.exp(1j * pd_), axis=1))))

    harm, perc = librosa.decompose.hpss(mag)
    hnr = float(np.mean(harm ** 2) / (np.mean(perc ** 2) + 1e-10))

    mfccs = librosa.feature.mfcc(y=waveform, sr=sr, n_mfcc=13, n_fft=n_fft)
    rms   = librosa.feature.rms(y=waveform, hop_length=hop, frame_length=n_fft)[0]
    
    # Extract spectral flatness
    flatness = librosa.feature.spectral_flatness(y=waveform, hop_length=hop, n_fft=n_fft)[0]

    vec = np.hstack([
        [float(f0.mean()), float(f0.std()), flux, phase_coh, hnr,
         float(rms.mean()), float(rms.std()), float(flatness.mean()), float(flatness.std()), duration],
        np.mean(mfccs, axis=1),
        np.std(mfccs,  axis=1),
    ])
    return np.nan_to_num(vec.astype(np.float32), nan=0.0, posinf=0.0, neginf=0.0)

## Metadata Features

One-hot language + text length / word count / unique-char ratio.
Fitted on train only - unseen languages at test map to all-zeros (`handle_unknown="ignore"`).

In [ ]:
METADATA_NAMES = ["text_length", "word_count", "unique_char_ratio"]

def _meta_cols(records):
    langs  = [str(r.get("language") or "unknown") for r in records]
    texts  = [str(r.get("text") or "") for r in records]
    text_f = np.array([[len(t), len(t.split()), len(set(t))/(len(t)+1)]
                        for t in texts], dtype=np.float32)
    return np.array(langs, dtype=object).reshape(-1,1), text_f

try:
    enc = OneHotEncoder(handle_unknown="ignore", sparse_output=False, dtype=np.float32)
except TypeError:
    enc = OneHotEncoder(handle_unknown="ignore", sparse=False, dtype=np.float32)
scaler = StandardScaler()

lang_tr, txt_tr = _meta_cols(train_records)
lang_te, txt_te = _meta_cols(test_records)

train_meta = np.hstack([enc.fit_transform(lang_tr), scaler.fit_transform(txt_tr)]).astype(np.float32)
test_meta  = np.hstack([enc.transform(lang_te),     scaler.transform(txt_te)    ]).astype(np.float32)

meta_names = [f"language={c}" for c in enc.categories_[0]] + METADATA_NAMES
print("Metadata feature dims:", train_meta.shape[1])

## Extract Handcrafted Features - Once (Deterministic, No Model)

In [ ]:
import os
from joblib import Parallel, delayed

def _hc_row(w_i16, dur):
    return extract_handcrafted(np.asarray(w_i16, dtype=np.float32) / 32767.0, dur)

def parallel_handcrafted(waves, durs, desc):
    rows = Parallel(n_jobs=os.cpu_count(), batch_size=32)(
        delayed(_hc_row)(np.asarray(waves[i]), durs[i])
        for i in tqdm(range(len(durs)), desc=desc))
    return np.vstack(rows).astype(np.float32)

if os.path.exists("train_hc.npy") and os.path.exists("test_hc.npy"):
    print("Loading cached handcrafted features from disk...")
    train_hc = np.load("train_hc.npy")
    test_hc  = np.load("test_hc.npy")
else:
    print("Extracting handcrafted features (parallel, from waveform cache)...")
    train_hc = parallel_handcrafted(train_waves, train_durs, "train handcrafted")
    np.save("train_hc.npy", train_hc)
    test_hc  = parallel_handcrafted(test_waves, test_durs, "test handcrafted")
    np.save("test_hc.npy", test_hc)

print("train_hc:", train_hc.shape, "  test_hc:", test_hc.shape)

## Wav2Vec2 + LoRA Model

LoRA adapts the transformer attention projections (Q, K, V, out) with ~3M trainable parameters
instead of fine-tuning the full backbone. The CNN feature encoder is kept frozen throughout.

Embeddings = mean-pool + std-pool over time on `[last_hidden_state ‖ mid-layer hidden state]`
→ `4 × hidden_size` dims (4096 for the indicwav2vec-hindi LARGE backbone, 3072 for a base one).
The dimensions are derived from the model config at load time - never hardcoded.
The std captures temporal variability - synthetic speech is often unnaturally smooth.

In [ ]:
# Load processor and base model with automatic fallback if gated model fails to download
from transformers import AutoConfig

try:
    print(f"Attempting to load processor for: {SPEECH_MODEL_NAME}")
    processor = Wav2Vec2Processor.from_pretrained(SPEECH_MODEL_NAME)
    print(f"Attempting to load model weights for: {SPEECH_MODEL_NAME}")
    _ = Wav2Vec2Model.from_pretrained(SPEECH_MODEL_NAME)
except Exception as e:
    print(f"\n[WARNING] Failed to load '{SPEECH_MODEL_NAME}': {e}")
    SPEECH_MODEL_NAME = "apoorva-ak/indicwav2vec_base"
    fallback_processor = "facebook/wav2vec2-base-960h"
    print(f"[LOG] Falling back to public, non-gated model: {SPEECH_MODEL_NAME} with processor: {fallback_processor}\n")
    processor = Wav2Vec2Processor.from_pretrained(fallback_processor)

# Derive dimensions from the actual backbone instead of hardcoding wav2vec2-base sizes.
# indicwav2vec-hindi is a LARGE checkpoint (hidden_size=1024, 24 layers), not base (768, 12).
_backbone_cfg = AutoConfig.from_pretrained(SPEECH_MODEL_NAME)
EMBED_DIM = _backbone_cfg.hidden_size
MID_LAYER = _backbone_cfg.num_hidden_layers // 2
print(f"[LOG] Backbone: hidden_size={EMBED_DIM}, layers={_backbone_cfg.num_hidden_layers}, "
      f"mid layer={MID_LAYER}, final embedding dim={EMBED_DIM * 4}")

class DeepfakeDetector(nn.Module):
    """Wav2Vec2 + LoRA backbone with a binary classification head."""

    def __init__(self):
        super().__init__()
        base = Wav2Vec2Model.from_pretrained(SPEECH_MODEL_NAME)
        base.feature_extractor._freeze_parameters()   # CNN encoder stays frozen
        if GRAD_CHECKPOINT:
            # non-reentrant checkpointing: reentrant fails when only LoRA params require grad
            base.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})

        lora_cfg = LoraConfig(
            r=LORA_RANK, lora_alpha=LORA_ALPHA,
            target_modules=["q_proj", "k_proj", "v_proj", "out_proj"],  # all attention projections
            lora_dropout=LORA_DROPOUT, bias="none",
        )
        self.backbone = get_peft_model(base, lora_cfg)
        # mean+std pooling over [final layer | mid layer] -> EMBED_DIM * 4
        self.head     = nn.Linear(EMBED_DIM * 4, 1)
        self.drop     = nn.Dropout(0.1)

    def get_embeddings(self, input_values, attention_mask=None):
        # autocast must live INSIDE forward: nn.DataParallel runs each replica in its
        # own thread and autocast state is thread-local
        with torch.autocast("cuda", enabled=USE_AMP and input_values.is_cuda):
            out = self.backbone(input_values, attention_mask=attention_mask,
                                output_hidden_states=True)
            h_final = out.last_hidden_state              # (B, T, EMBED_DIM) - semantic
            h_mid   = out.hidden_states[MID_LAYER]       # (B, T, EMBED_DIM) - acoustic textures
            h = torch.cat([h_final, h_mid], dim=-1)      # (B, T, EMBED_DIM*2)
        h = h.float()                                    # pool in fp32: std is precision-sensitive
        return torch.cat([h.mean(dim=1), h.std(dim=1)], dim=1)   # (B, EMBED_DIM*4)

    def forward(self, input_values, attention_mask=None, return_embeddings=False):
        emb = self.get_embeddings(input_values, attention_mask)
        if return_embeddings:
            return emb
        return self.head(self.drop(emb)).squeeze(-1)   # (B,) fp32 logits


def _make_model():
    m = DeepfakeDetector().to(DEVICE)
    trainable = sum(p.numel() for p in m.parameters() if p.requires_grad)
    total     = sum(p.numel() for p in m.parameters())
    print(f"  Trainable params: {trainable:,}  /  Total: {total:,}"
          f"  ({100*trainable/total:.1f}%)")
    if torch.cuda.device_count() > 1:
        print(f"  [LOG] Wrapping model in nn.DataParallel across {torch.cuda.device_count()} GPUs.")
        m = nn.DataParallel(m)
    return m

## Training and Embedding-Extraction Functions

In [ ]:
from torch.optim.lr_scheduler import OneCycleLR

def _batch_inputs(waves, rows):
    """int16 cache rows -> normalized processor tensors on DEVICE."""
    batch_w = [np.asarray(waves[r], dtype=np.float32) / 32767.0 for r in rows]
    inp = processor(batch_w, sampling_rate=TARGET_SR, return_tensors="pt", padding=True)
    iv = inp.input_values.to(DEVICE)
    am = inp.get("attention_mask")
    if am is not None:
        am = am.to(DEVICE)
    return iv, am


def fine_tune(model, waves, sample_idx, labels_arr, desc):
    """Supervised LoRA fine-tuning with fp16 AMP, reading from the waveform cache."""
    pos_w = torch.tensor(
        [max(1.0, float((labels_arr == 0).sum()) / float((labels_arr == 1).sum() + 1e-9))],
        device=DEVICE)
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_w)
    optimizer = torch.optim.AdamW(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=LORA_LR, weight_decay=0.01)

    n = len(sample_idx)
    steps_per_epoch = (n + FINETUNE_BATCH - 1) // FINETUNE_BATCH
    scheduler = OneCycleLR(optimizer, max_lr=LORA_LR,
                           total_steps=N_EPOCHS * steps_per_epoch, pct_start=0.1)
    scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)

    order = np.arange(n)
    model.train()

    for epoch in range(N_EPOCHS):
        np.random.shuffle(order)
        total_loss, steps = 0.0, 0
        for start in tqdm(range(0, n, FINETUNE_BATCH),
                          desc=f"{desc} epoch {epoch+1}/{N_EPOCHS}", leave=False):
            sel   = order[start:start + FINETUNE_BATCH]
            iv, am = _batch_inputs(waves, sample_idx[sel])
            lbl_t  = torch.tensor(labels_arr[sel], dtype=torch.float32, device=DEVICE)

            optimizer.zero_grad(set_to_none=True)
            logits = model(iv, am)              # autocast lives inside forward()
            loss   = criterion(logits.float(), lbl_t)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()

            total_loss += loss.item(); steps += 1

        avg_loss = total_loss / steps
        print(f"  [LOG] {desc} | Epoch {epoch+1}/{N_EPOCHS} completed | Avg Loss: {avg_loss:.6f}")
    return model


@torch.inference_mode()
def extract_embeddings(model, waves, sample_idx, desc):
    """Return (embeddings (N, EMBED_DIM*4), sigmoid probabilities (N,))."""
    model.eval()
    core = model.module if hasattr(model, "module") else model
    emb_parts, prob_parts = [], []
    for start in tqdm(range(0, len(sample_idx), EXTRACT_BATCH), desc=desc):
        rows = sample_idx[start:start + EXTRACT_BATCH]
        iv, am = _batch_inputs(waves, rows)
        emb  = model(iv, am, return_embeddings=True)     # DataParallel-safe
        prob = torch.sigmoid(core.head(emb).squeeze(-1)) # dropout inactive in eval
        emb_parts.append(emb.float().cpu().numpy())
        prob_parts.append(prob.float().cpu().numpy())
    return (np.vstack(emb_parts).astype(np.float32),
            np.concatenate(prob_parts).astype(np.float32))

## 5-Fold OOF Fine-Tuning + Fold-Ensemble Test Features

Each fold trains on 4 folds, then extracts embeddings **and** NN probabilities for:
- the held-out fold (leak-free OOF features for XGBoost)
- the full test set (cheap: only 2.6k clips per fold)

```
Fold k held out -> fine-tune on the other 4 -> extract held-out fold k + test set
stitch held-out parts -> OOF features for 100% of train, zero leakage
average the 5 test-set passes -> free 5-model ensemble
```

The old separate "6th model trained on all data" is gone: averaging the 5 fold models
on the test set costs almost nothing, saves a full fine-tuning run (~20% of total
compute), and is a stronger predictor than any single model. Each fold's LoRA adapter
(~12 MB) and head are kept; the best-AUC fold is exported to ONNX at the end.

In [ ]:
import os
import json

expected_shape = (len(train_records), EMBED_DIM * 4)

oof_embeddings, oof_nn_prob = None, None
if os.path.exists("oof_embeddings.npy") and os.path.exists("oof_nn_prob.npy"):
    cached = np.load("oof_embeddings.npy")
    if cached.shape == expected_shape:
        print("Loading existing OOF embeddings from disk...")
        oof_embeddings = cached
        oof_nn_prob    = np.load("oof_nn_prob.npy")
if oof_embeddings is None:
    print("Starting a fresh OOF run (no cache, or stale cache discarded).")
    for stale in ("oof_embeddings.npy", "oof_nn_prob.npy",
                  "completed_folds.json", "fold_aucs.json"):
        if os.path.exists(stale):
            os.remove(stale)
    oof_embeddings = np.zeros(expected_shape, dtype=np.float32)
    oof_nn_prob    = np.zeros(len(train_records), dtype=np.float32)

completed_folds, fold_aucs = [], {}
if os.path.exists("completed_folds.json"):
    with open("completed_folds.json") as f:
        completed_folds = json.load(f)
if os.path.exists("fold_aucs.json"):
    with open("fold_aucs.json") as f:
        fold_aucs = json.load(f)
# a fold only counts as completed if its per-fold test artifacts survived
completed_folds = [f for f in completed_folds
                   if os.path.exists(f"test_emb_fold{f}.npy")
                   and os.path.exists(f"fold{f}_head.pt")]
if completed_folds:
    print("Completed folds loaded from disk:", completed_folds)

skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
test_idx_all = np.arange(len(test_records))

for fold, (tr_idx, val_idx) in enumerate(skf.split(np.arange(len(train_records)), train_labels)):
    if fold in completed_folds:
        print(f"\nSkipping Fold {fold+1}/{N_FOLDS} (already completed and cached)")
        continue

    print(f"\n{'='*60}")
    print(f"  FOLD {fold+1}/{N_FOLDS}  |  train {len(tr_idx)}  val {len(val_idx)}")
    print(f"{'='*60}")

    model = _make_model()
    model = fine_tune(model, train_waves, tr_idx, train_labels[tr_idx], desc=f"Fold {fold+1}")

    # held-out fold: leak-free OOF features
    fold_emb, fold_prob = extract_embeddings(model, train_waves, val_idx,
                                             desc=f"Fold {fold+1} val embeddings")
    oof_embeddings[val_idx] = fold_emb
    oof_nn_prob[val_idx]    = fold_prob

    fold_auc = roc_auc_score(train_labels[val_idx], fold_prob)
    fold_aucs[str(fold)] = float(fold_auc)
    print(f"  Fold {fold+1} held-out NN AUC: {fold_auc:.5f}")

    # test set: one pass per fold, averaged later into a 5-model ensemble
    test_emb, test_prob = extract_embeddings(model, test_waves, test_idx_all,
                                             desc=f"Fold {fold+1} test embeddings")
    np.save(f"test_emb_fold{fold}.npy", test_emb)
    np.save(f"test_prob_fold{fold}.npy", test_prob)

    # LoRA adapters are ~12 MB: keep every fold's; the best is exported to ONNX later
    model_module = model.module if hasattr(model, "module") else model
    model_module.backbone.save_pretrained(f"fold{fold}_adapter")
    torch.save(model_module.head.state_dict(), f"fold{fold}_head.pt")

    # Save progress after each fold
    np.save("oof_embeddings.npy", oof_embeddings)
    np.save("oof_nn_prob.npy", oof_nn_prob)
    completed_folds.append(fold)
    with open("completed_folds.json", "w") as f:
        json.dump(completed_folds, f)
    with open("fold_aucs.json", "w") as f:
        json.dump(fold_aucs, f)

    del model; gc.collect()
    if DEVICE.type == "cuda": torch.cuda.empty_cache()

    print(f"  Fold {fold+1} stored: {fold_emb.shape}")

print(f"\nOOF embeddings complete: {oof_embeddings.shape}")
if len(completed_folds) == N_FOLDS:
    print("OOF NN AUC (entire train set):", round(roc_auc_score(train_labels, oof_nn_prob), 5))

## Train XGBoost on OOF Embeddings

`[OOF embeddings (4 x hidden_size)] + [handcrafted (36-d)] + [metadata]` -> XGBoost.

The AUC here is honest: each embedding was produced by a model that never saw that example,
and XGBoost is scored on a held-out slice it didn't train on.

In [ ]:
X_train = np.hstack([oof_embeddings, train_hc, train_meta]).astype(np.float32)
y_train = train_labels

all_feat_names = (
    [f"wav2vec2_mean_{i:03d}" for i in range(EMBED_DIM * 2)]
    + [f"wav2vec2_std_{i:03d}"  for i in range(EMBED_DIM * 2)]
    + HANDCRAFTED_NAMES + meta_names
)

print("X_train:", X_train.shape, "  features:", len(all_feat_names))
print("Positive rate:", y_train.mean().round(4))

# ── Phase 1: early stopping to find best tree count ────────────────────────────
# split by index so the NN OOF probabilities can be scored on the same val rows
idx_tr, idx_val = train_test_split(
    np.arange(len(y_train)), test_size=XGB_VAL_SIZE, random_state=SEED, stratify=y_train)
X_tr, X_val = X_train[idx_tr], X_train[idx_val]
y_tr, y_val = y_train[idx_tr], y_train[idx_val]

pos_w = max(1.0, float((y_tr == 0).sum()) / float((y_tr == 1).sum() + 1e-9))

early_model = XGBClassifier(
    **XGB_PARAMS, scale_pos_weight=pos_w,
    early_stopping_rounds=XGB_EARLY_STOP, device=XGB_DEVICE)
early_model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=100)

best_n = (getattr(early_model, "best_iteration", None) or 0) + 1
print(f"Best iteration: {early_model.best_iteration} → using {best_n} trees")

# ── Phase 2: refit on ALL training data with validated tree count ───────────────
final_params = dict(XGB_PARAMS, n_estimators=best_n)
xgb_model = XGBClassifier(**final_params, scale_pos_weight=pos_w, device=XGB_DEVICE)
xgb_model.fit(X_train, y_train, verbose=False)
print("Refit on full training data - done.")

# ── Save Laptop-Accessible Features & Preprocessors ─────────────────────
print("[LOG] Saving training datasets and preprocessing states for offline laptop use...")
np.save("X_train.npy", X_train)
np.save("y_train.npy", y_train)

with open("all_feat_names.json", "w") as f:
    json.dump(all_feat_names, f)

import pickle
with open("preprocessors.pkl", "wb") as f:
    pickle.dump({"encoder": enc, "scaler": scaler}, f)
print("[LOG] Saved X_train.npy, y_train.npy, all_feat_names.json, and preprocessors.pkl.")

In [ ]:
val_probs = early_model.predict_proba(X_val)[:, 1]
val_preds = (val_probs >= 0.5).astype(int)

# ── Blend weight: XGBoost vs NN ensemble, selected on the validation split ─────
# w = weight on the NN probability; w=0 is pure XGBoost, w=1 pure NN
nn_val_probs = oof_nn_prob[idx_val]
blend_aucs = {w: float(roc_auc_score(y_val, (1 - w) * val_probs + w * nn_val_probs))
              for w in (0.0, 0.25, 0.5, 0.75, 1.0)}
BLEND_W = max(blend_aucs, key=blend_aucs.get)
print("Blend AUCs (w = NN weight):", {k: round(v, 5) for k, v in blend_aucs.items()})
print(f"Selected BLEND_W = {BLEND_W}")

metrics = {
    "roc_auc":         float(roc_auc_score(y_val, val_probs)),
    "pr_auc":          float(average_precision_score(y_val, val_probs)),
    "accuracy_at_0.5": float(accuracy_score(y_val, val_preds)),
    "f1_at_0.5":       float(f1_score(y_val, val_preds, zero_division=0)),
    "nn_val_auc":      blend_aucs[1.0],
    "blend_w":         float(BLEND_W),
    "blend_auc":       blend_aucs[BLEND_W],
    "best_iteration":  int(getattr(early_model, "best_iteration", 0) or 0),
    "n_estimators":    int(best_n),
    "val_examples":    int(len(y_val)),
    "positive_rate":   float(y_val.mean()),
}
print(json.dumps(metrics, indent=2))
with open("metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)
print("[LOG] Saved validation metrics to metrics.json.")

## Fold-Ensemble Test Features

Collect the per-fold test artifacts saved during the OOF loop. NN probabilities are
averaged directly (standard ensemble). Embeddings stay **per-fold**: XGBoost was trained
on single-model OOF embeddings, so it predicts on each fold's embeddings separately and
the 5 probability vectors are averaged - this keeps the feature distribution XGBoost
sees at test time identical to what it saw at training time.

In [ ]:
fold_test_embs, fold_test_probs = [], []
for f in sorted(completed_folds):
    e = np.load(f"test_emb_fold{f}.npy")
    p = np.load(f"test_prob_fold{f}.npy")
    assert e.shape == (len(test_records), EMBED_DIM * 4), \
        f"stale fold {f} test embeddings {e.shape} - rerun the OOF cell"
    fold_test_embs.append(e)
    fold_test_probs.append(p)

assert len(fold_test_embs) == N_FOLDS, \
    f"only {len(fold_test_embs)}/{N_FOLDS} folds completed - rerun the OOF cell"

nn_test_prob = np.mean(fold_test_probs, axis=0).astype(np.float32)
print(f"NN ensemble test probabilities from {len(fold_test_probs)} folds:", nn_test_prob.shape)

## Predictions → submission.csv

In [ ]:
# XGBoost predicts on each fold's test embeddings separately, probabilities averaged
xgb_fold_probs = []
for e in fold_test_embs:
    X_test_f = np.hstack([e, test_hc, test_meta]).astype(np.float32)
    xgb_fold_probs.append(xgb_model.predict_proba(X_test_f)[:, 1])
xgb_test_prob = np.mean(xgb_fold_probs, axis=0)

# artifacts for offline laptop use (mean embedding version) + trained XGBoost model
print("[LOG] Saving test feature dataset and XGBoost model to disk...")
np.save("X_test.npy", np.hstack(
    [np.mean(fold_test_embs, axis=0), test_hc, test_meta]).astype(np.float32))
xgb_model.save_model("xgb_model.json")

# final prediction: XGB/NN blend with the weight validated in the metrics cell
test_probs = (1.0 - BLEND_W) * xgb_test_prob + BLEND_W * nn_test_prob
print(f"Blending with BLEND_W={BLEND_W} (0 = pure XGBoost, 1 = pure NN ensemble)")

submission = pd.DataFrame({
    "id":     [str(test_ds[i]["id"]) for i in range(len(test_ds))],
    "is_tts": test_probs,
})
submission.to_csv("submission.csv", index=False)
print("Wrote submission.csv:", submission.shape)
submission.head(10)

## Top Feature Importances

In [ ]:
imp   = xgb_model.feature_importances_
imp_order = np.argsort(-imp)[:20]
pd.DataFrame({
    "feature":    [all_feat_names[i] for i in imp_order],
    "importance": imp[imp_order],
}).style.bar(subset=["importance"], color="#4C72B0")

## Export to ONNX for Deployment

Pick the fold with the best held-out AUC, merge its LoRA adapter into a clean base
model, attach its trained head, and export to ONNX.

In [ ]:
# ─── ONNX Export for Deployment ─────────────────────────────────────
import os
import json
import torch
from peft import PeftModel

print("[LOG] Exporting fine-tuned model to ONNX for deployment...")

with open("fold_aucs.json") as f:
    _fold_aucs = {int(k): v for k, v in json.load(f).items()}
best_fold = max(_fold_aucs, key=_fold_aucs.get)
adapter_dir, head_path = f"fold{best_fold}_adapter", f"fold{best_fold}_head.pt"
print(f"Best fold: {best_fold + 1} (held-out AUC {_fold_aucs[best_fold]:.5f}) -> exporting {adapter_dir}")

for req in (adapter_dir, head_path):
    if not os.path.exists(req):
        raise FileNotFoundError(f"'{req}' not found - run the OOF fine-tuning cell first.")

# Rebuild a clean backbone (no PEFT wrapper, no DataParallel, no gradient
# checkpointing) and merge the saved LoRA adapter into it for export.
_export_base    = Wav2Vec2Model.from_pretrained(SPEECH_MODEL_NAME)
_peft_backbone  = PeftModel.from_pretrained(_export_base, adapter_dir)
merged_backbone = _peft_backbone.merge_and_unload()
print("Successfully loaded and merged LoRA adapter weights into base model.")

class ExportModel(nn.Module):
    """Merged backbone + trained head - same pooling as DeepfakeDetector, ONNX-traceable."""
    def __init__(self, backbone):
        super().__init__()
        self.backbone = backbone
        self.head = nn.Linear(EMBED_DIM * 4, 1)

    def forward(self, input_values, attention_mask=None):
        out = self.backbone(input_values, attention_mask=attention_mask, output_hidden_states=True)
        h   = torch.cat([out.last_hidden_state, out.hidden_states[MID_LAYER]], dim=-1)
        emb = torch.cat([h.mean(dim=1), h.std(dim=1)], dim=1)
        return self.head(emb).squeeze(-1)

model_to_export = ExportModel(merged_backbone).to(DEVICE)
model_to_export.head.load_state_dict(torch.load(head_path, map_location=DEVICE))
model_to_export.eval()

# ship the preprocessing config alongside the adapter for deployment
processor.save_pretrained(adapter_dir)

# Create a dummy input (B=1, T=16000 * 5) to trace the graph
dummy_input = torch.zeros((1, MAX_SAMPLES), dtype=torch.float32).to(DEVICE)
dummy_mask  = torch.ones((1, MAX_SAMPLES), dtype=torch.long).to(DEVICE)

onnx_path = "deepfake_detector.onnx"

try:
    torch.onnx.export(
        model_to_export,
        (dummy_input, dummy_mask),
        onnx_path,
        input_names=["input_values", "attention_mask"],
        output_names=["logits"],
        dynamic_axes={
            "input_values": {0: "batch_size", 1: "sequence_length"},
            "attention_mask": {0: "batch_size", 1: "sequence_length"},
            "logits": {0: "batch_size"}
        },
        opset_version=17,
        do_constant_folding=True
    )
    print(f"[SUCCESS] Exported fine-tuned model to ONNX: {onnx_path}")
except Exception as e:
    print(f"[ERROR] ONNX export failed: {e}")
    raise e
